In [2]:
import asyncio
import json
import os
import uuid
from typing import Annotated, Any, List, Literal, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, interrupt

In [4]:
from swift.env import SUPABASE_ACCESS_TOKEN, CAL_API_KEY


load_dotenv()

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────


# Tools that require human approval before executing
PROTECTED_TOOLS = {
    "create_project",
    "delete_project",
    "execute_sql",        
    "apply_migration",
}

MCP_CONFIG = {
    # "supabase": {
    #     "transport": "stdio",
    #     "command": "npx",
    #     "args": [
    #         "-y",
    #         "@supabase/mcp-server-supabase@latest",
    #         "--access-token", SUPABASE_ACCESS_TOKEN
    #     ],
    # },
    "calcom": {
    "transport": "stdio",
      "command": "npx",
      "args": ["@calcom/cal-mcp@latest"],
      "env": {
        "CAL_API_KEY": CAL_API_KEY
      }
    }
}


# ─────────────────────────────────────────────────────────────────
# STATE
# ─────────────────────────────────────────────────────────────────

class State(TypedDict):
    messages:  Annotated[list, add_messages]
    yolo_mode: bool   # if True, skip all human review


# ─────────────────────────────────────────────────────────────────
# GRAPH
# ─────────────────────────────────────────────────────────────────

async def build_graph():
    # ── Connect to Supabase MCP and load tools ─────────────────────
    client = MultiServerMCPClient(MCP_CONFIG)
    tools  = await client.get_tools()

    print(f"  Loaded {len(tools)} tools from Supabase MCP")
    print(f"  Protected: {PROTECTED_TOOLS}\n")

    llm = ChatGroq(
        # base_url="https://openrouter.ai/api/v1",
        model="openai/gpt-oss-120b",
        temperature=0.1,
        streaming=True,
    ).bind_tools(tools, parallel_tool_calls=True)

    # ── NODE 1: agent ──────────────────────────────────────────────
    async def agent_node(state: State):
        system = SystemMessage(content=(
            "You are a helpful Supabase assistant. "
            "Help the user manage their Supabase projects, databases, and schemas. "
            "For read operations just proceed. "
            "Write/destructive operations will be reviewed by the user before running."
        ))
        response = await llm.ainvoke([system] + state["messages"])
        return {"messages": [response]}

    # ── NODE 2: human_review_node ──────────────────────────────────
    #
    # Sits between agent and tools. For every protected tool call in
    # the last AIMessage it calls interrupt() once, collects a decision,
    # then routes accordingly.
    #
    # Decisions:
    #   continue              → run the tool exactly as the agent planned
    #   update  {new_args}    → run the tool with merged argument overrides
    #   feedback <message>    → skip the tool, send your message to the agent
    #                           so it can understand why and try again
    #   reject                → skip the tool, tell the agent it was rejected
    #
    # After reviewing ALL protected calls in this batch:
    #   - If any were rejected/feedbacked AND there are still approved calls
    #     to run → go to tools_node AND return feedback to agent in same step
    #   - If only feedback/rejects and nothing left to run → go back to agent
    #   - Otherwise → go to tools_node with all approved + safe calls

    async def human_review_node(state: State) -> Command[Literal["agent_node", "tools_node"]]:
        last_message = state["messages"][-1]

        if not isinstance(last_message, AIMessage) or not last_message.tool_calls:
            return Command(goto="agent_node")

        protected_calls = [
            c for c in last_message.tool_calls if c["name"] in PROTECTED_TOOLS
        ]
        safe_calls = [
            c for c in last_message.tool_calls if c["name"] not in PROTECTED_TOOLS
        ]

        feedback_messages: list[ToolMessage] = []
        approved_calls:    list[dict]        = []

        for call in protected_calls:
            # ── Pause graph execution and wait for human input ──
            decision: dict = interrupt({
                "tool_call": call,
                "message":   f"Agent wants to call '{call['name']}'",
                "args":      call["args"],
                "options":   "continue | update {<new_args>} | feedback <message> | reject",
            })

            action = decision.get("action", "continue")

            if action == "continue":
                approved_calls.append(call)

            elif action == "update":
                # Merge provided arg overrides into the original args
                new_args     = decision.get("args", {})
                updated_call = {**call, "args": {**call["args"], **new_args}}
                approved_calls.append(updated_call)

            elif action == "feedback":
                # Skip the tool; inject reviewer message so agent can react
                feedback_messages.append(ToolMessage(
                    content=decision.get("message", "The user provided feedback on this tool call."),
                    name=call["name"],
                    tool_call_id=call["id"],
                ))

            elif action == "reject":
                feedback_messages.append(ToolMessage(
                    content=(
                        "This tool call was rejected by the user. "
                        "Ask the user how they would like to proceed."
                    ),
                    name=call["name"],
                    tool_call_id=call["id"],
                ))

        # ── Determine where to go next ─────────────────────────────
        calls_to_run = approved_calls + safe_calls

        # Rebuild the AIMessage so tool_calls matches tool results exactly
        updated_ai = AIMessage(
            content=last_message.content,
            tool_calls=calls_to_run,
            id=last_message.id,
        )

        if calls_to_run:
            # There are tools to run — go to tools_node.
            # Attach any feedback messages so the agent also sees them
            # after the tools finish.
            return Command(
                goto="tools_node",
                update={"messages": [updated_ai] + feedback_messages},
            )
        else:
            # Everything was rejected/feedbacked — nothing to run.
            # Go back to agent with the feedback so it can respond.
            return Command(
                goto="agent_node",
                update={"messages": [updated_ai] + feedback_messages},
            )

    # ── NODE 3: tools_node ─────────────────────────────────────────
    # ToolNode runs all approved tool calls in parallel and handles
    # errors per-tool, returning ToolMessage(status="error") on failure
    # so the agent can recover gracefully.
    def handle_tool_error(error: Exception) -> str:
        return json.dumps({"status": "error", "error": str(error)})

    tools_node = ToolNode(tools, handle_tool_errors=handle_tool_error)

    # ── ROUTER: after agent, decide where to go ────────────────────
    def should_continue(state: State) -> str:
        last = state["messages"][-1]

        if not isinstance(last, AIMessage) or not last.tool_calls:
            return END

        # yolo_mode: skip human review entirely
        if state.get("yolo_mode", False):
            return "tools_node"

        # Any protected call in this batch? Route through review node
        if any(c["name"] in PROTECTED_TOOLS for c in last.tool_calls):
            return "human_review_node"

        # All safe — go straight to tools
        return "tools_node"

    # ── Wire up the graph ──────────────────────────────────────────
    graph = StateGraph(State)
    graph.add_node("agent_node",        agent_node)
    graph.add_node("human_review_node", human_review_node)
    graph.add_node("tools_node",        tools_node)

    graph.add_edge(START, "agent_node")
    graph.add_conditional_edges(
        "agent_node",
        should_continue,
        {
            "human_review_node": "human_review_node",
            "tools_node":        "tools_node",
            END:                 END,
        },
    )
    graph.add_edge("tools_node", "agent_node")
    # human_review_node uses Command(goto=...) to route itself — no edge needed

    return graph.compile(checkpointer=InMemorySaver()), client


# ─────────────────────────────────────────────────────────────────
# STREAMING RUNNER
# ─────────────────────────────────────────────────────────────────

async def run(user_message: str, graph, config: dict):
    """
    Run one conversation turn.
    Streams AI tokens, tool call names/args, and tool results live.
    Pauses on interrupts and collects human decisions before resuming.
    """
    current_input: Any = {"messages": [HumanMessage(content=user_message)]}

    while True:
        interrupt_payload = None
        in_ai_stream      = False

        async for chunk in graph.astream(
            current_input,
            config=config,
            stream_mode=["messages", "updates", "values"],
            version="v2",
        ):
            ctype = chunk.get("type")

            if ctype == "messages":
                msg, _meta = chunk["data"]

                # Stream AI text tokens as they arrive
                if isinstance(msg, AIMessage) and msg.content:
                    if not in_ai_stream:
                        print("\n🤖 Assistant: ", end="", flush=True)
                        in_ai_stream = True
                    print(msg.content, end="", flush=True)

                # Stream tool call name + args as the LLM generates them
                if isinstance(msg, AIMessage) and msg.tool_call_chunks:
                    for tc in msg.tool_call_chunks:
                        if tc.get("name"):
                            print(f"\n⚙️  Calling: \033[93m{tc['name']}\033[0m ", end="", flush=True)
                        if tc.get("args"):
                            print(f"\033[2m{tc['args']}\033[0m", end="", flush=True)

                # Print tool results
                if isinstance(msg, ToolMessage):
                    if in_ai_stream:
                        print()
                        in_ai_stream = False
                    try:
                        pretty = json.dumps(json.loads(msg.content), indent=2)
                    except Exception:
                        pretty = msg.content
                    print(f"\n🔧 Tool result:\n\033[2m{pretty}\033[0m")

            elif ctype == "values":
                interrupts = chunk.get("interrupts", [])
                if interrupts:
                    interrupt_payload = interrupts[0]

        if in_ai_stream:
            print()

        # ── Human review prompt ────────────────────────────────────
        if interrupt_payload is not None:
            payload   = interrupt_payload.value
            call      = payload["tool_call"]
            args      = call["args"]
            args_json = json.dumps(args, indent=2)
            one_liner = json.dumps(args)

            print("\n" + "═"*60)
            print(f"  🛡  {payload['message']}")
            print("─"*60)
            print("  Current args (copy & edit for update):")
            for line in args_json.splitlines():
                print(f"    {line}")
            print("─"*60)
            print("  continue              → run as-is")
            print("  reject                → cancel, tell agent")
            print("  feedback <message>    → skip + tell agent why")
            print()
            print("  To update args, copy the JSON above, change what")
            print("  you need, then type:  update <edited JSON>")
            print()
            print(f"  Example: update {one_liner}")
            print("═"*60)

            raw   = input("\n  Decision › ").strip()
            lower = raw.lower()

            if lower in ("continue", ""):
                decision = {"action": "continue"}

            elif lower.startswith("update"):
                rest = raw[6:].strip()
                try:
                    new_args = json.loads(rest)
                except json.JSONDecodeError:
                    print("  ⚠ Couldn't parse JSON — running as-is")
                    new_args = {}
                decision = {"action": "update", "args": new_args}

            elif lower.startswith("feedback"):
                decision = {"action": "feedback", "message": raw[8:].strip()}

            elif lower == "reject":
                decision = {"action": "reject"}

            else:
                print("  ⚠ Unknown input — defaulting to continue")
                decision = {"action": "continue"}

            current_input = Command(resume=decision)

        else:
            break


# ─────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────

async def main():
    print("""
╔══════════════════════════════════════════════════════╗
║          🗄  Supabase Agent                          ║
╚══════════════════════════════════════════════════════╝

Safe tools run automatically.
Protected tools (create/delete/execute_sql) pause for review.
Type 'exit' to quit. Type 'yolo' to toggle auto-approve.
""")
    print("  Connecting to Supabase MCP server...")
    graph, client = await build_graph()
    config        = {"configurable": {"thread_id": f"t-{uuid.uuid4().hex[:6]}"}}
    yolo          = False
    print("  Ready!\n")

    while True:
        user_input = input("You: ").strip()

        if not user_input:
            continue

        if user_input.lower() == "exit":
            print("Bye!")
            break

        if user_input.lower() == "yolo":
            yolo = not yolo
            print(f"  Yolo mode: {'ON ⚡ skipping all reviews' if yolo else 'OFF 🛡 reviews enabled'}\n")
            continue

        await run(user_input, graph, config)


# Works both as a script and in Jupyter (await main())
if __name__ == "__main__":
    # asyncio.run(main())
    await main()


╔══════════════════════════════════════════════════════╗
║          🗄  Supabase Agent                          ║
╚══════════════════════════════════════════════════════╝

Safe tools run automatically.
Protected tools (create/delete/execute_sql) pause for review.
Type 'exit' to quit. Type 'yolo' to toggle auto-approve.

  Connecting to Supabase MCP server...
  Loaded 9 tools from Supabase MCP
  Protected: {'execute_sql', 'apply_migration', 'delete_project', 'create_project'}

  Ready!



APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01k3kbp4tzf00vt0g49px4akgv` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 11517, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [5]:
client = MultiServerMCPClient(MCP_CONFIG)
tools  = await client.get_tools()

In [6]:
tools

[StructuredTool(name='getBookings', description='Get all bookings', args_schema={'type': 'object', 'properties': {'params': {'properties': {'cal-api-version': {'default': '2024-08-13', 'type': 'string'}, 'status': {'type': 'array', 'items': {'enum': ['upcoming', 'recurring', 'past', 'cancelled', 'unconfirmed'], 'type': 'string'}}, 'attendeeEmail': {'type': 'string'}, 'attendeeName': {'type': 'string'}, 'eventTypeIds': {'type': 'string'}, 'eventTypeId': {'type': 'string'}, 'teamsIds': {'type': 'string'}, 'teamId': {'type': 'string'}, 'afterStart': {'type': 'string'}, 'beforeEnd': {'type': 'string'}, 'afterUpdatedAt': {'type': 'string'}, 'beforeUpdatedAt': {'type': 'string'}, 'sortStart': {'enum': ['asc', 'desc'], 'type': 'string'}, 'sortEnd': {'enum': ['asc', 'desc'], 'type': 'string'}, 'sortCreated': {'enum': ['asc', 'desc'], 'type': 'string'}, 'sortUpdatedAt': {'enum': ['asc', 'desc'], 'type': 'string'}, 'take': {'type': 'number'}, 'skip': {'type': 'number'}, 'Authorization': {'type':